# 01 · Yarış hafta sonunu tanı

**Hedef:** gerçek bir hafta sonunun seans, sürücü ve tur tablolarını okuyabilmek.

Altyapı hazır. Senin görevin tabloyu anlamak ve sürücü başına benzersiz tur sayısını yazmak. Bu defter tahmin üretmez. Örnek: 2024 Bahreyn.

## Kurulum

Üst sağdan **WeAreChecking (.venv)** kernelini seç. Önceki Olist ortamını seçme.
İndirme gerekiyorsa `BASLA.md` içindeki fetch komutunu terminalde çalıştır.
Hücreleri sırayla `Shift+Enter` ile çalıştır.

In [ ]:
import importlib
import pandas as pd
from IPython.display import display
from wearechecking.paths import project_root
from wearechecking.data import latest_snapshot
from wearechecking import student
from wearechecking.learning import check_lap_counts

ROOT = project_root()
YEAR, ROUND = 2024, 1
print('WeAreChecking ortamı yüklendi.')

## Adımlar

### 1. Snapshot'ı aç

Snapshot, indirilen verinin değiştirmeden sakladığımız kopyasıdır.
`observed_at` veriyi bizim ne zaman gördüğümüzdür; yarış tarihi değildir.
Veri yoksa bu hücre fetch komutunu gösterir; gerçek veri yerine örnek veri koymaz.

In [ ]:
snapshot = latest_snapshot(YEAR, ROUND, root=ROOT)
print('Etkinlik:', snapshot.manifest['event_name'])
print('Kaynak:', snapshot.manifest['source'])
print('İndirme zamanı:', snapshot.manifest['observed_at'])
print('Veri türü:', snapshot.manifest['provenance_mode'])

### 2. Seans takvimini oku

Bir satır **bir seans**. FP1/FP2/FP3 antrenman, Q sıralama, R yarış.
Sprint hafta sonunda SQ ve S olabilir; seansları elle sabitlemiyoruz.
Saatler UTC. Önce kendi cümlenle seans sırasını anlat.

In [ ]:
schedule = snapshot.schedule()
display(schedule)
expected = set(schedule.session_code)
loaded = set(snapshot.manifest['sessions'])
assert loaded == expected, f'Eksik seanslar: {expected - loaded}. Tüm hafta sonunu fetch ile indir.'
print(f'{len(loaded)} / {len(expected)} seans yüklü.')

### 3. Tur tablosunu oku

Bir satır **bir sürücünün bir tur kaydıdır**. `DriverNumber` seans içindeki sürücü numarası; kalıcı sürücü kimliği değildir.
`LapTime` boş olabilir: ilk tur, eksik ölçüm gibi nedenler var. Boş değeri sıfır yapma.

**Dur ve bana yaz:** DriverNumber ile LapNumber arasındaki fark ne?

In [ ]:
laps = snapshot.table('FP1', 'laps')
display(laps.sort_values(['DriverNumber', 'LapNumber'])[['DriverNumber', 'LapNumber', 'LapTime', 'is_pit_out_lap']].head(8))
print('Satır sayısı:', len(laps))
print('Sürücü numarası sayısı:', laps.DriverNumber.nunique())
print('Eksik tur süresi:', int(laps.LapTime.isna().sum()))

### 4. Diğer tabloların amacı

Katılımcı: bu seansa kim girdi? Stint: aynı lastik setiyle hangi turlar atıldı? Hava: o anda ne ölçüldü?
Bu tabloları indirmek, hepsini hemen modele vermek anlamına gelmez.

In [ ]:
overview = pd.DataFrame([
    {'tablo': name, 'satır': len(snapshot.table('FP1', name))}
    for name in ['participants', 'laps', 'stints', 'weather']
])
display(overview)

### 5. Senin alıştırman: lap_counts

`src/wearechecking/student.py` içinde yalnız `lap_counts` fonksiyonunu düzenle.

Çıktı kolonları: **driver_number, lap_count**. Her sürücü tek satır.
Tekrarlı `(DriverNumber, LapNumber)` kayıtlarını bir kez say; boş/sıfır tur numaralarını sayma.

İpuçları:
1. Geçerli tur satırlarını seç.
2. `drop_duplicates` için hangi iki kolon birlikte tekil olmalı?
3. `groupby` ile sürücülere ayır ve satırları say.
4. Çıktı kolonlarını istenen adlara dönüştür.

Tüm çözümü bir kerede yazma; birlikte adım adım ilerleyeceğiz.

In [ ]:
importlib.reload(student)
answer = student.lap_counts(laps.copy())
print(check_lap_counts(answer, laps))
if answer is not None:
    display(answer.head(10))

## Kontroller

- Gerçek verinin kaynağı ve indirme zamanı görünür.
- Takvimdeki bütün seanslar snapshot'ta var.
- Bir tur kaydını bir sürücü kaydından ayırabiliyorum.
- `lap_counts` kontrolü **TAMAM** diyor.
- Çok tur atan sürücünün mutlaka daha hızlı olmadığını açıklayabiliyorum.

Son iki madde bitmeden hedef 1 tamamlanmış sayılmaz.

## Sonraki adım

Yanıtını paylaştığında `lap_counts` fonksiyonunu birlikte tamamlayacağız. Sonra 02 defteri.

Kaynaklar: [OpenF1](https://openf1.org/docs/) · [Jolpica](https://github.com/jolpica/jolpica-f1/blob/main/docs/README.md).